## Install requirements

In [1]:
!pip install -r ../requirements.txt


  Using cached scikit_learn-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (10.8 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.8.0
    Uninstalling scikit-learn-1.8.0:
      Successfully uninstalled scikit-learn-1.8.0

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import torch
import sys
sys.path.append('../')

from utils.event_decoder import EventDecoder
from validate_birdset import load_model
from utils.transform import ValTransform
from checkpoints import DT, MT, LT
import librosa
import soundfile as sf
import pandas as pd

SAMPLE_RATE = 32000
FILEPATH = "./XC973025.mp3"
DEVICE = "cpu" # change to gpu if cuda is available

/home/bellafkir/Documents/sa4birds/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
!wget -O ebird_taxonomy_v2022.csv !wget -O ebird_taxonomy_v2022.csv https://www.birds.cornell.edu/clementschecklist/wp-content/uploads/2022/12/ebird_taxonomy_v2022.csv

--2026-03-15 15:46:59--  http://!wget/
Resolving !wget (!wget)... failed: Name or service not known.
wget: unable to resolve host address ‘!wget’
--2026-03-15 15:46:59--  https://www.birds.cornell.edu/clementschecklist/wp-content/uploads/2022/12/ebird_taxonomy_v2022.csv
Resolving www.birds.cornell.edu (www.birds.cornell.edu)... 141.193.213.10, 141.193.213.11
Connecting to www.birds.cornell.edu (www.birds.cornell.edu)|141.193.213.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2344912 (2.2M) [text/csv]
Saving to: ‘ebird_taxonomy_v2022.csv’

ebird_taxonomy_v202 100%[===================>]   2.24M  --.-KB/s    in 0.07s   

2026-03-15 15:46:59 (29.9 MB/s) - ‘ebird_taxonomy_v2022.csv’ saved [2344912/2344912]

FINISHED --2026-03-15 15:46:59--
Total wall clock time: 0.2s
Downloaded: 1 files, 2.2M in 0.07s (29.9 MB/s)


In [2]:
!wget -O XC973025.mp3 https://xeno-canto.org/973025/download

--2026-03-15 15:47:01--  https://xeno-canto.org/973025/download
Resolving xeno-canto.org (xeno-canto.org)... 145.136.248.176
Connecting to xeno-canto.org (xeno-canto.org)|145.136.248.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [audio/mpeg]
Saving to: ‘XC973025.mp3’

XC973025.mp3            [ <=>                ]   1.25M  --.-KB/s    in 0.09s   

2026-03-15 15:47:02 (13.7 MB/s) - ‘XC973025.mp3’ saved [1311882]



In [7]:
taxonomy = pd.read_csv('ebird_taxonomy_v2022.csv')
code_to_scientific = taxonomy.set_index("SPECIES_CODE")["SCI_NAME"].to_dict()

In [8]:

ckpt = LT.get('ALL', [])[0]

# Load the trained model and its configuration from the checkpoint
model, cfg = load_model('../' + ckpt, device=DEVICE)

# pick a temperature for the attention 
model.attention.temperature = 2.0

# Create the validation transform pipeline
# This handles spectrogram generation and preprocessing
transform = ValTransform(
        config=cfg,
        train=False,
        event_decoder=EventDecoder(extracted_interval=cfg.event_decoder.val.extracted_interval),
)

# Reverse the label map so we can convert predicted class index -> label name
label_map = { v:k for k,v in cfg.train.label_map.items()}

# Read metadata about the audio file (samplerate, duration, etc.)
file_info = sf.info(FILEPATH)
sr = file_info.samplerate

# Load the first 7 seconds of the audio file
audio, sr = sf.read(FILEPATH, start=0, stop=7*sr)


# If audio is multi-channel (e.g., stereo), convert it to mono
if audio.ndim != 1:
    audio = audio.swapaxes(1, 0)
    audio = librosa.to_mono(audio)

# Resample audio if it does not match the model's required sample rate
if sr != SAMPLE_RATE:
    audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    sr = SAMPLE_RATE

# Convert the waveform to a tensor and compute spectrogram features
# Shape transformation: (samples) -> (batch=1, channel=1, samples)
fbank_features = transform.get_spectrogram(torch.tensor(audio).unsqueeze(0).unsqueeze(0).float())

# Pad the spectrogram so it matches the expected model input size
fbank_features = transform.pad(fbank_features)

# Normalize the features using dataset mean and std from the config
fbank_features = (fbank_features - cfg.frontend.mean) / ( cfg.frontend.std * 2)

# Run inference without computing gradients (faster and uses less memory)
with torch.no_grad():
    preds = model(fbank_features.to(DEVICE))

    # Get the top-1 predicted class
    top2 = preds.topk(2)
    print(f"{code_to_scientific[label_map[int(top2.indices[0][0])]]}: {top2.values[0][0]}")
    print(f"{code_to_scientific[label_map[int(top2.indices[0][1])]]}: {top2.values[0][1]}")


2026-03-15 15:47:51,816 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-15 15:47:51,935 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-15 15:47:51,939 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)


Milvus milvus: 0.9462558627128601
Columba palumbus: 0.6704452037811279
